In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.switch_backend('TkAgg')
import seaborn as sns
import mplcursors

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve
)

## **Loading the data**

In [2]:
import pandas as pd

dataset = pd.read_csv("cleaned_spam.csv")


In [3]:
print("Dataset preview:\n")
print(dataset.head(), "\n")

print("Columns:")
print(dataset.columns)

Dataset preview:

   label                                      clean_message
0      0  go jurong point crazy available bugis n great ...
1      0                            ok lar joking wif u oni
2      1  free entry number wkly comp win fa cup final t...
3      0                u dun say early hor u c already say
4      0             nah think goes usf lives around though 

Columns:
Index(['label', 'clean_message'], dtype='object')


## **Cleaning the data**

In [4]:
print("Number of rows having missing value")
print(dataset['clean_message'].isnull().sum())

Number of rows having missing value
6


In [5]:
cleaned_dataset = dataset.dropna(subset=['clean_message','label'])

## **Defining X and Y**

In [6]:
X = cleaned_dataset['clean_message'].astype(str)
Y = cleaned_dataset['label']

## **Splitting the data into Train and Test data**

In [7]:
X_train , X_test, Y_train, Y_test = train_test_split(X, Y , test_size = 0.2 , stratify = Y , random_state = 42)

## **Comparing the models Logistic Regression, MultinomialNB and LinearSVM**

In [8]:
result = []
all_prediction = {}

for n in [1000, 3000, 5000, 10000]:
    print(f"\n==============================")
    print(f"Testing max_features = {n}")
    print(f"==============================")
    
    half = n // 2
    
    char_vectorizer = TfidfVectorizer(max_features=half, analyzer='char', ngram_range=(3,5))
    word_vectorizer = TfidfVectorizer(max_features=half, analyzer='word', ngram_range=(1,2))
    
    combined = FeatureUnion([
        ('char', char_vectorizer),
        ('word', word_vectorizer)
    ])
    
    X_train_vec = combined.fit_transform(X_train)
    X_test_vec = combined.transform(X_test)
    
    print("Shape:", X_train_vec.shape)
    
    models = {
        "Naive Bayes": MultinomialNB(),
        "Linear SVM": LinearSVC(),
        "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced')
    }
    
    for name, model in models.items():
        
        model.fit(X_train_vec, Y_train)
        Y_pred = model.predict(X_test_vec)
        
        all_prediction[(n , name )] = Y_pred
        
        report = classification_report(Y_test, Y_pred, output_dict=True)
        
        print(f"Model: {name} | max_features: {n}")
        print(f"F1-score: {report['1']['f1-score']:.4f}")
        
        result.append({
            "max_features": n,
            "model": name,
            "precision_spam": report['1']['precision'],
            "recall_spam": report['1']['recall'],
            "f1_spam": report['1']['f1-score']
        })

results_table = pd.DataFrame(result)

print("\nFINAL RESULTS:")
print(results_table)


Testing max_features = 1000
Shape: (4136, 1000)
Model: Naive Bayes | max_features: 1000
F1-score: 0.8127
Model: Linear SVM | max_features: 1000
F1-score: 0.9313
Model: Logistic Regression | max_features: 1000
F1-score: 0.8495

Testing max_features = 3000
Shape: (4136, 3000)
Model: Naive Bayes | max_features: 3000
F1-score: 0.8421
Model: Linear SVM | max_features: 3000
F1-score: 0.9502
Model: Logistic Regression | max_features: 3000
F1-score: 0.8759

Testing max_features = 5000
Shape: (4136, 5000)
Model: Naive Bayes | max_features: 5000
F1-score: 0.8707
Model: Linear SVM | max_features: 5000
F1-score: 0.9579
Model: Logistic Regression | max_features: 5000
F1-score: 0.8819

Testing max_features = 10000
Shape: (4136, 10000)
Model: Naive Bayes | max_features: 10000
F1-score: 0.9130
Model: Linear SVM | max_features: 10000
F1-score: 0.9498
Model: Logistic Regression | max_features: 10000
F1-score: 0.8850

FINAL RESULTS:
    max_features                model  precision_spam  recall_spam   f1

### Model Summary

Linear SVM performed best due to its effectiveness on high-dimensional text data, achieving the highest F1-score. Increasing max_features improved performance up to 5000, after which gains were minimal. Combining character and word n-grams enhanced detection of both patterns and variations, making the model more robust.

## **Analysing the 3 Models using Graph**

In [9]:
df = pd.DataFrame(result)

models = df['model'].unique()

plt.figure(figsize = (15,6))

lines = []

for m in models:
    data = df[df['model'] == m]
    
    l1, = plt.plot(data['max_features'], data['f1_spam'], marker = 'o', label = f"{m} - F1")
    l2, = plt.plot(data['max_features'], data['recall_spam'], marker = 'x', linestyle = '--', label = f"{m} - Recall")
    l3, = plt.plot(data['max_features'], data['precision_spam'], marker = 's', linestyle = '-.', label = f"{m} - Precision")
    lines.extend([l1, l2, l3])

cursor = mplcursors.cursor(lines, hover = True)

@cursor.connect("add")
def on_add(sel):
    x = sel.target[0]
    y = sel.target[1]
    sel.annotation.set_text(f"x = {x}\ny = {y:.3f}")
 
plt.xlabel("Max Features")        
plt.ylabel("Score")
plt.title("Model-Wise Performance vs Max Features")

plt.legend(bbox_to_anchor = (1.02,1), loc = 'upper left')
plt.grid()
plt.tight_layout(rect=[0, 0, 0.8, 1])
plt.show()

### Line Graph Interpretation:

The line graph shows how precision, recall, and F1-score change with increasing max_features for each model.
Linear SVM consistently maintains the highest performance, especially in F1-score across all feature sizes.
Performance improves initially but stabilizes after 5000 features, indicating feature saturation.

## **Final Comparison of Models**

In [10]:
best_models = results_table.loc[
    results_table.groupby('model')['f1_spam'].idxmax()
]

plt.figure()

bars = plt.bar(best_models['model'], best_models['f1_spam'])

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height,f'{height:.3f}', ha='center', va='bottom')

plt.xlabel("Model")
plt.ylabel("F1 Score")
plt.title("Best F1 Score Comparison of Models")

plt.ylim(0.8, 1)   
plt.grid(axis='y')

plt.tight_layout()
plt.show()

## **Listing Confusion Matrix**

In [11]:
for (n, name) , Y_pred in all_prediction.items():
    cm = confusion_matrix(Y_test, Y_pred)
    print(f"\nConfusion Matrix for {name} with max_features = {n}")
    print(cm)


Confusion Matrix for Naive Bayes with max_features = 1000
[[847  55]
 [  4 128]]

Confusion Matrix for Linear SVM with max_features = 1000
[[894   8]
 [ 10 122]]

Confusion Matrix for Logistic Regression with max_features = 1000
[[862  40]
 [  5 127]]

Confusion Matrix for Naive Bayes with max_features = 3000
[[858  44]
 [  4 128]]

Confusion Matrix for Linear SVM with max_features = 3000
[[897   5]
 [  8 124]]

Confusion Matrix for Logistic Regression with max_features = 3000
[[871  31]
 [  5 127]]

Confusion Matrix for Naive Bayes with max_features = 5000
[[868  34]
 [  4 128]]

Confusion Matrix for Linear SVM with max_features = 5000
[[898   4]
 [  7 125]]

Confusion Matrix for Logistic Regression with max_features = 5000
[[873  29]
 [  5 127]]

Confusion Matrix for Naive Bayes with max_features = 10000
[[884  18]
 [  6 126]]

Confusion Matrix for Linear SVM with max_features = 10000
[[898   4]
 [  9 123]]

Confusion Matrix for Logistic Regression with max_features = 10000
[[874  2

In [12]:
print(Y.value_counts())

label
0    4510
1     660
Name: count, dtype: int64


## **Cross Validation**

In [13]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

f1_scores = []

for train_index, test_index in skf.split(X, Y):
    
    X_train_cv, X_test_cv = X.iloc[train_index], X.iloc[test_index]
    Y_train_cv, Y_test_cv = Y.iloc[train_index], Y.iloc[test_index]
    
   
    char_vectorizer = TfidfVectorizer(max_features=2500, analyzer='char', ngram_range=(3,5))
    word_vectorizer = TfidfVectorizer(max_features=2500, analyzer='word', ngram_range=(1,2))
    
    combined_cv = FeatureUnion([
        ('char', char_vectorizer),
        ('word', word_vectorizer)
    ])
    
    X_train_vec = combined_cv.fit_transform(X_train_cv)
    X_test_vec = combined_cv.transform(X_test_cv)
    
    model = LinearSVC()
    model.fit(X_train_vec, Y_train_cv)
    
    Y_pred_cv = model.predict(X_test_vec)
    
    f1 = f1_score(Y_test_cv, Y_pred_cv)
    f1_scores.append(f1)

print("F1 Scores for each fold:", f1_scores)
print("Mean F1 Score:", np.mean(f1_scores))

F1 Scores for each fold: [0.9163346613545816, 0.972972972972973, 0.9375, 0.9486166007905138, 0.9490196078431372]
Mean F1 Score: 0.9448887685922411


### Observations

- The model was tested on 5 different splits of data
- F1 scores are similar across all folds
- This shows the model is stable

### Conclusion

The model performance is consistent and not dependent on a single train-test split.

## **Final Model Training**

In [14]:
n = 5000
half = n //2

char_vectorizer = TfidfVectorizer(max_features=half, analyzer='char', ngram_range=(3,5))
word_vectorizer = TfidfVectorizer(max_features=half, analyzer='word', ngram_range=(1,2))

combined = FeatureUnion([('char', char_vectorizer), ('word', word_vectorizer)])

X_train_vec = combined.fit_transform(X_train)
X_test_vec = combined.transform(X_test)

final_model = LinearSVC()
final_model.fit(X_train_vec, Y_train)

print("Final Model Trained.")

Final Model Trained.


## **Threshold and F1-Score**

In [15]:
score = final_model.decision_function(X_test_vec)

best_f1 = 0
best_threshold = 0

for threshold in [-0.5, -0.3, -0.2, -0.1, 0, 0.1, 0.2 , 0.3, 0.4, 0.5]:
    Y_pred_new = (score> threshold).astype(int)
    
    report_new = classification_report(Y_test, Y_pred_new, output_dict=True)
    f1_new = report_new['1']['f1-score']
    
    print(f"Threshold {threshold}: F1 Score = {f1_new:.4f}")
    
    if f1_new > best_f1:
        best_f1 = f1_new
        best_threshold = threshold

print(f"Best F1 Score: {best_f1}")
print(f"Best Threshold: {best_threshold}")

Threshold -0.5: F1 Score = 0.8746
Threshold -0.3: F1 Score = 0.9377
Threshold -0.2: F1 Score = 0.9474
Threshold -0.1: F1 Score = 0.9545
Threshold 0: F1 Score = 0.9579
Threshold 0.1: F1 Score = 0.9575
Threshold 0.2: F1 Score = 0.9453
Threshold 0.3: F1 Score = 0.9486
Threshold 0.4: F1 Score = 0.9355
Threshold 0.5: F1 Score = 0.9268
Best F1 Score: 0.9578544061302682
Best Threshold: 0


## **Classification Report for Final Model**

In [16]:
score = final_model.decision_function(X_test_vec)
Y_pred_final = (score > best_threshold).astype(int)

print(classification_report(Y_test, Y_pred_final))

              precision    recall  f1-score   support

           0       0.99      1.00      0.99       902
           1       0.97      0.95      0.96       132

    accuracy                           0.99      1034
   macro avg       0.98      0.97      0.98      1034
weighted avg       0.99      0.99      0.99      1034



## **Confusion Matrix of Final Model**

In [17]:
cm_final = confusion_matrix(Y_test, Y_pred_final)
print(f"\nConfusion Matrix for Final Model (Linear SVM, max_features = 5000)") 
print(cm_final)


Confusion Matrix for Final Model (Linear SVM, max_features = 5000)
[[898   4]
 [  7 125]]


## **Figure illustrating Confusion Matrix**

In [18]:
cm_new = confusion_matrix(Y_test, Y_pred_final)

sns.heatmap(cm_new, annot=True, fmt='d', xticklabels=['Ham','Spam'],
            yticklabels=['Ham','Spam'])

plt.title("Confusion Matrix (Final Model)")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.tight_layout()
plt.show()

## **Precision Recall Curve**

In [19]:
precision, recall, threshold = precision_recall_curve(Y_test, score)
plt.plot(recall, precision, label="PR Curve", linewidth=2)
plt.axhline(y=sum(Y_test)/len(Y_test), linestyle='--', label='Baseline')

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve (Final Model)")
plt.grid()

plt.legend()
plt.show()

### Interpretation

The precision-recall curve shows a strong balance between precision and recall,
indicating that the model performs well in identifying spam messages while minimizing false predictions.

## **Error Analysis**

In [20]:
error_df = pd.DataFrame({
    "message": X_test,
    "actual": Y_test,
    "predicted": Y_pred_final
})

# False Positives
fp = error_df[(error_df['actual'] == 0) & (error_df['predicted'] == 1)]

# False Negatives
fn = error_df[(error_df['actual'] == 1) & (error_df['predicted'] == 0)]

print("False Positives:\n", fp.head(10))
print("\nFalse Negatives:\n", fn.head(10))

False Positives:
                                                 message  actual  predicted
3810                                         noice text       0          1
680   sorry trouble u buy number dad number number n...       0          1
3586       yep fine number number number number ice age       0          1
966   yun ah ubi one say wan call tomorrow call numb...       0          1

False Negatives:
                                                 message  actual  predicted
4497  check choose babe videos sms shsex netun fgksl...       1          0
2302  babe u want dont u baby im nasty thing number ...       1          0
3947  missed call alert numbers called left message ...       1          0
5170  dear customer kyc pending update avoid account...       1          0
3741                                ringtoneking number       1          0
3888  simpsons movie released july number name band ...       1          0
5174                  parcel hold pay rs number release       1

### Observations

- False Positives:
  Normal messages classified as spam.
  Some false positives like "noice text" are very short and lack clear context

- False Negatives:
  Spam messages classified as normal.
  Likely due to:
  - short messages
  - obfuscated words (e.g., fr33, w1n)
  
  Some false negatives like "kyc pending" looks like real service message

### Conclusion

This project developed a spam detection model using TF-IDF features with both character and word n-grams. Multiple models were compared, and Linear SVM achieved the best performance with an F1-score of around 0.95.

Evaluation using confusion matrix and precision-recall analysis showed that the model performs well overall, though it struggles with ambiguous and short messages.

Overall, the model is accurate, stable, and effective for spam detection, with scope for improvement using more advanced techniques.